# 01 · Evaluation task set

**Goal:** a fixed set of broken C# programs to compare an LLM **without tools**, **with only the compiler**, and **with all MCP tools**.
Programs come from Project 4's held-out test set (never used to train the fixer model) plus Project 3's real build failures.

## 1. Setup

In [1]:
# Cell 3: Setup.
import sys                                              # To add the project folder to Python's import path
from pathlib import Path                                # File paths that work on any OS

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # Project folder, wherever it lives
sys.path.insert(0, str(ROOT))                           # So our .py files can be imported

import pandas as pd                                     # Tables
from csharp import compile_code                         # Check that every broken program really fails

P4 = ROOT.parent / "csharp-fix-lora"                    # Project 4 folder (next to this one)
TASKS_FILE = ROOT / "evals" / "tasks.jsonl"             # Output: the evaluation tasks
MAX_CHARS = 3_000                                       # The fixer tool's limit: keep tasks it can handle
print("P4 test set found:", (P4 / "data" / "processed" / "test.jsonl").exists())

P4 test set found: True


## 2. Pick the tasks
Single-mistake examples only, at most one per seed program (more variety), a fixed random seed (repeatable).

In [2]:
# Cell 5: Select 10 + 10 + 4 from the P4 test set, plus the 4 real P3 failures.
test = pd.read_json(P4 / "data" / "processed" / "test.jsonl", lines=True)   # 407 held-out examples
test = test[~test["kinds"].str.contains(r"\+") & (test["broken"].str.len() <= MAX_CHARS)]   # One mistake, short enough
PLAN = [("remove_using", "missing_using", 10), ("wrong_member", "wrong_api", 10),
        ("drop_semicolon", "syntax", 2), ("drop_closing", "syntax", 2)]   # (operator, category, how many)

rows = []
for op, category, n in PLAN:
    part = test[test["kinds"] == op].drop_duplicates("seed_id")   # At most one example per seed program
    for r in part.sample(min(n, len(part)), random_state=7).to_dict("records"):   # Repeatable random pick
        rows.append({"id": r["id"], "category": category, "source": "p4_test", "broken": r["broken"], "fixed": r["fixed"]})

real = pd.read_json(P4 / "data" / "processed" / "real_errors.jsonl", lines=True)   # Project 3's real build failures
for r in real.to_dict("records"):
    rows.append({"id": "real/" + r["id"].split("/")[-1] + "_" + r["id"].split("/")[0].replace("results_", ""),
                 "category": "real_llm_error", "source": "p3_real", "broken": r["broken"], "fixed": ""})   # No reference answer

tasks = pd.DataFrame(rows)
print(tasks["category"].value_counts().to_dict())       # How many per category

{'missing_using': 10, 'wrong_api': 10, 'syntax': 4, 'real_llm_error': 4}


## 3. Verify and save
Every broken program must really fail to compile today; record its first error code.

In [3]:
# Cell 7: Compile every broken program (about 1 minute), keep only real failures, save.
first_errors, keep = [], []
for code in tasks["broken"]:
    r = compile_code(code)                              # Real dotnet build
    keep.append(not r.succeeded)                        # Must fail
    first_errors.append(r.errors[0]["code"] if r.errors else "")
tasks["first_error"] = first_errors
tasks = tasks[keep].reset_index(drop=True)

TASKS_FILE.parent.mkdir(exist_ok=True)                  # Make sure evals/ exists
tasks.to_json(TASKS_FILE, orient="records", lines=True, force_ascii=False)
print(f"Saved {len(tasks)} tasks -> {TASKS_FILE}")
print(pd.crosstab(tasks["category"], tasks["first_error"]))   # Which errors each category produces

Saved 28 tasks -> d:\AI Projects\dotnet-dev-mcp\evals\tasks.jsonl
first_error     CS0103  CS0246  CS1002  CS1003  CS1026  CS1061  CS1501  CS1503
category                                                                      
missing_using        3       7       0       0       0       0       0       0
real_llm_error       1       0       0       0       0       2       0       1
syntax               0       0       1       1       2       0       0       0
wrong_api            0       0       0       0       0       9       1       0
